# Embed Clues

Generate sentence-transformer embeddings for all clue texts and correct responses stored in the DuckDB database.
Each run performs a **full rewrite** — the embeddings table is cleared and rebuilt from scratch.

In [ ]:
%load_ext autoreload
%autoreload 2

import duckdb
from sentence_transformers import SentenceTransformer

from jt3.db import ensure_schema
from jt3.embeddings import DEFAULT_MODEL

conn = duckdb.connect("../data/jt3.duckdb")
ensure_schema(conn)
model_name = "minishlab/potion-base-2M"
# model_name = DEFAULT_MODEL
device = "cuda"

st_model = SentenceTransformer(model_name, device=device)

%load_ext sql
%sql conn --alias duckdb
print(f"Model:  {model_name}")
print(f"Device: {st_model.device}")

## Check current state

Show how many clues are in the DB and how many already have embeddings.

In [ ]:
%%sql status <<
SELECT
    (SELECT COUNT(*) FROM clues) AS total_clues,
    (SELECT COUNT(*) FROM clues WHERE correct_response IS NOT NULL) AS total_responses

In [ ]:
df = status.PolarsDataFrame()
total = df["total_clues"][0] + df["total_responses"][0]
print(f"Total to embed: {total}")

## Run embeddings

Embeds all clues and inserts into the embeddings table (replacing any existing rows).

In [ ]:
%%sql clues_df <<
SELECT game_id, round_index, category_index, clue_id, text, correct_response
FROM clues

In [ ]:
import polars as pl
from tqdm.notebook import tqdm

clues = clues_df.PolarsDataFrame()

entries = []
texts = []
for row in tqdm(clues.iter_rows(named=True), total=len(clues), desc="Building entries"):
    entries.append((row["game_id"], row["round_index"], row["category_index"], row["clue_id"], "text", row["text"]))
    texts.append(row["text"])
        
    if row["correct_response"] is not None:
        entries.append((row["game_id"], row["round_index"], row["category_index"], row["clue_id"], "correct_response", row["correct_response"]))
        texts.append(row["correct_response"])

print(f"To embed: {len(entries)}")

vectors = st_model.encode(texts, show_progress_bar=True)

new_embeddings = pl.DataFrame({
    "game_id": [e[0] for e in entries],
    "round_index": [e[1] for e in entries],
    "category_index": [e[2] for e in entries],
    "clue_id": [e[3] for e in entries],
    "field": [e[4] for e in entries],
    "source_text": [e[5] for e in entries],
    "embedding": [v.tolist() for v in vectors],
    "model_name": [model_name] * len(entries),
})
print(f"Built DataFrame with {len(new_embeddings)} rows, ready to insert.")

In [ ]:
%%sql
DELETE FROM embeddings;
INSERT INTO embeddings SELECT * FROM new_embeddings

## Verify completeness

Confirm that every embeddable field now has an embedding.

In [ ]:
%%sql verify <<
SELECT
    (SELECT COUNT(*) FROM clues) AS total_clues,
    (SELECT COUNT(*) FROM clues WHERE correct_response IS NOT NULL) AS total_responses,
    (SELECT COUNT(*) FROM embeddings WHERE field = 'text') AS text_embedded,
    (SELECT COUNT(*) FROM embeddings WHERE field = 'correct_response') AS resp_embedded

In [ ]:
df = verify.PolarsDataFrame()
total_expected = df["total_clues"][0] + df["total_responses"][0]
total_embedded = df["text_embedded"][0] + df["resp_embedded"][0]

print(f"Clue texts embedded: {df['text_embedded'][0]}/{df['total_clues'][0]}")
print(f"Responses embedded:  {df['resp_embedded'][0]}/{df['total_responses'][0]}")
print(f"Total:               {total_embedded}/{total_expected}")
print(f"Complete:            {'Yes' if total_embedded == total_expected else 'No'}")